# 05a · Image Basics — How Computers See

Images are just numbers. A 256×256 color photo is a 256×256×3 array of integers (0–255).  
This notebook covers how to load, manipulate, transform, and augment images for deep learning.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
import torchvision
import torchvision.transforms as T
from torchvision.datasets import CIFAR10, MNIST
from torch.utils.data import DataLoader

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["image.interpolation"] = "nearest"

## How Computers See Images

An image is a 3D array: **height × width × channels**.  
Each pixel has 3 values (Red, Green, Blue) ranging from 0 to 255.

| What you see | What the computer sees |
|---|---|
| A photo of a cat | A 224×224×3 array of numbers |
| "Red" | `[255, 0, 0]` |
| "White" | `[255, 255, 255]` |

In [ ]:
dataset = CIFAR10(root="./data", train=True, download=True)
img, label = dataset[49]
class_names = dataset.classes

img_array = np.array(img)
print(f"Image shape: {img_array.shape}  (H × W × C)")
print(f"Pixel value range: [{img_array.min()}, {img_array.max()}]")
print(f"Label: {class_names[label]}")

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
axes[0].imshow(img_array)
axes[0].set_title(f"Original — {class_names[label]}")

channel_names = ["Red", "Green", "Blue"]
cmaps = ["Reds", "Greens", "Blues"]
for i in range(3):
    axes[i + 1].imshow(img_array[:, :, i], cmap=cmaps[i])
    axes[i + 1].set_title(f"{channel_names[i]} Channel")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\nTop-left 4×4 pixel block (Red channel):")
print(img_array[:4, :4, 0])

---
## Image as a Tensor

Different libraries use different axis orders:

| Library | Format | Shape Example |
|---------|--------|---------------|
| NumPy / PIL / OpenCV | H × W × C | `(32, 32, 3)` |
| PyTorch | **C × H × W** | `(3, 32, 32)` |

PyTorch puts channels first because convolutions operate on channels.

In [ ]:
numpy_img = np.array(img)
print(f"NumPy/PIL shape (H×W×C): {numpy_img.shape}")

tensor_img = T.ToTensor()(img)
print(f"PyTorch shape  (C×H×W): {tensor_img.shape}")
print(f"Value range after ToTensor: [{tensor_img.min():.1f}, {tensor_img.max():.1f}]")

manual_tensor = torch.from_numpy(numpy_img).permute(2, 0, 1).float() / 255.0
print(f"\nManual conversion matches: {torch.allclose(tensor_img, manual_tensor)}")

back_to_numpy = tensor_img.permute(1, 2, 0).numpy()
print(f"Back to NumPy shape: {back_to_numpy.shape}")

---
## Basic Operations

Resize, crop, flip, and rotate — the bread and butter of image manipulation.

In [ ]:
img_large = img.resize((128, 128), Image.BILINEAR)

operations = {
    "Original": img_large,
    "Resize 64×64": img_large.resize((64, 64)),
    "Center Crop": T.CenterCrop(80)(img_large),
    "Horizontal Flip": T.RandomHorizontalFlip(p=1.0)(img_large),
    "Vertical Flip": T.RandomVerticalFlip(p=1.0)(img_large),
    "Rotate 45°": T.RandomRotation((45, 45))(img_large),
}

fig, axes = plt.subplots(1, len(operations), figsize=(18, 3))
for ax, (name, result) in zip(axes, operations.items()):
    ax.imshow(np.array(result))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## Color Transformations

Convert to grayscale, adjust brightness and contrast — all common preprocessing steps.

In [ ]:
color_ops = {
    "Original": img_large,
    "Grayscale": T.Grayscale(num_output_channels=3)(img_large),
    "Bright +50%": ImageEnhance.Brightness(img_large).enhance(1.5),
    "Bright −50%": ImageEnhance.Brightness(img_large).enhance(0.5),
    "High Contrast": ImageEnhance.Contrast(img_large).enhance(2.0),
    "Low Contrast": ImageEnhance.Contrast(img_large).enhance(0.3),
}

fig, axes = plt.subplots(1, len(color_ops), figsize=(18, 3))
for ax, (name, result) in zip(axes, color_ops.items()):
    ax.imshow(np.array(result))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## Normalization

Neural networks train faster when inputs are centered around 0 with unit variance.  
The **ImageNet** normalization values are used almost universally for pretrained models:

| Channel | Mean | Std |
|---------|------|-----|
| Red     | 0.485 | 0.229 |
| Green   | 0.456 | 0.224 |
| Blue    | 0.406 | 0.225 |

These were computed across the entire ImageNet dataset (~1.2M images).

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

raw_tensor = T.ToTensor()(img)
normalize = T.Normalize(mean=imagenet_mean, std=imagenet_std)
normalized = normalize(raw_tensor)

print("Before normalization:")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={raw_tensor[i].mean():.3f}, std={raw_tensor[i].std():.3f}")

print("\nAfter normalization:")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={normalized[i].mean():.3f}, std={normalized[i].std():.3f}")

def denormalize(tensor, mean, std):
    t = tensor.clone()
    for c in range(3):
        t[c] = t[c] * std[c] + mean[c]
    return t.clamp(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].imshow(raw_tensor.permute(1, 2, 0))
axes[0].set_title("Raw (0–1)")
axes[1].imshow(normalized.permute(1, 2, 0).clamp(0, 1))
axes[1].set_title("Normalized (clipped for display)")
axes[2].imshow(denormalize(normalized, imagenet_mean, imagenet_std).permute(1, 2, 0))
axes[2].set_title("De-normalized (recovered)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## Data Augmentation

Augmentation creates *variations* of training images so the model sees more diversity  
without needing more data. Each epoch, the same image looks slightly different.

Key augmentations in `torchvision.transforms`:
- `RandomCrop` — random sub-region
- `RandomHorizontalFlip` — mirror left↔right
- `RandomRotation` — small angle rotations
- `ColorJitter` — brightness, contrast, saturation, hue
- `RandomErasing` — randomly mask out a rectangle

In [ ]:
augmentations = {
    "Original": T.Compose([T.Resize(128)]),
    "RandomCrop": T.Compose([T.Resize(128), T.RandomCrop(96), T.Resize(128)]),
    "HorizontalFlip": T.Compose([T.Resize(128), T.RandomHorizontalFlip(p=1.0)]),
    "Rotation(±30°)": T.Compose([T.Resize(128), T.RandomRotation(30)]),
    "ColorJitter": T.Compose([T.Resize(128), T.ColorJitter(0.5, 0.5, 0.5, 0.1)]),
    "RandomErasing": T.Compose([T.Resize(128), T.ToTensor(), T.RandomErasing(p=1.0, scale=(0.1, 0.3))]),
}

fig, axes = plt.subplots(2, len(augmentations), figsize=(18, 6))
for row in range(2):
    for col, (name, transform) in enumerate(augmentations.items()):
        result = transform(img)
        if isinstance(result, torch.Tensor):
            result = result.permute(1, 2, 0).numpy()
        axes[row, col].imshow(np.array(result))
        if row == 0:
            axes[row, col].set_title(name, fontsize=10)
        axes[row, col].axis("off")

fig.suptitle("Same image, different random augmentations (2 runs)", fontsize=13)
plt.tight_layout()
plt.show()

---
## Compose Transforms — Full Augmentation Pipeline

`T.Compose` chains transforms sequentially. A typical training pipeline:

In [ ]:
train_transform = T.Compose([
    T.Resize(256),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Training pipeline (with augmentation):")
for i, t in enumerate(train_transform.transforms):
    print(f"  {i+1}. {t}")

print("\nValidation pipeline (deterministic):")
for i, t in enumerate(val_transform.transforms):
    print(f"  {i+1}. {t}")

augmented = train_transform(img)
print(f"\nOutput shape: {augmented.shape}")
print(f"Output range: [{augmented.min():.2f}, {augmented.max():.2f}]")

---
## Loading Datasets

**`torchvision.datasets`** has common datasets built in. For custom data, use **`ImageFolder`**.

In [ ]:
mnist_transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
mnist = MNIST(root="./data", train=True, download=True, transform=mnist_transform)
print(f"MNIST: {len(mnist)} images, shape: {mnist[0][0].shape}")

cifar_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
cifar = CIFAR10(root="./data", train=True, download=True, transform=cifar_transform)
print(f"CIFAR-10: {len(cifar)} images, shape: {cifar[0][0].shape}")

loader = DataLoader(cifar, batch_size=64, shuffle=True, num_workers=2)
batch_imgs, batch_labels = next(iter(loader))
print(f"\nBatch shape: {batch_imgs.shape}")
print(f"Labels shape: {batch_labels.shape}")

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))

raw_cifar = CIFAR10(root="./data", train=True, download=False)
for i in range(8):
    img_c, lbl = raw_cifar[i]
    axes[0, i].imshow(np.array(img_c))
    axes[0, i].set_title(raw_cifar.classes[lbl], fontsize=9)
    axes[0, i].axis("off")

raw_mnist = MNIST(root="./data", train=True, download=False)
for i in range(8):
    img_m, lbl = raw_mnist[i]
    axes[1, i].imshow(np.array(img_m), cmap="gray")
    axes[1, i].set_title(f"Label: {lbl}", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("CIFAR-10", fontsize=11)
axes[1, 0].set_ylabel("MNIST", fontsize=11)
plt.tight_layout()
plt.show()

---

### Key Takeaways

| Concept | Remember |
|---|---|
| Image = array | H × W × C for NumPy/PIL, **C × H × W** for PyTorch |
| `ToTensor()` | Converts PIL → tensor AND scales 0–255 → 0–1 |
| Normalization | Use ImageNet mean/std for pretrained models |
| Augmentation | Increases effective dataset size — use for training only |
| `Compose` | Chain transforms into a pipeline |
| `DataLoader` | Batches + shuffles + parallel loading |